# Lab 8 - PySpark i SQL, Partycjonowanie oraz Nowoczesna Hurtownia Danych Lakehouse Delta Lake

Celem niniejszego laboratorium jest poznanie zasad organizacji i przetwarzania danych w architekturze **Lakehouse** przy użyciu **PySpark SQL** oraz technologii **Delta Lake**.

Nowoczesny standard inżynierii danych: **Delta Lake**, który eliminuje wady Hive i wprowadza pełne transakcje ACID, wersjonowanie (Time Travel) oraz zaawansowaną organizację fizyczną plików.

---

### Agenda laboratorium:
1. **Konfiguracja środowiska:** Sesja `SparkSession` ze wsparciem dla Delta Lake (`DeltaCatalog`).
2. **Spark i SQL:** Praca z obiektami DataFrame, rejestracja widoków tymczasowych (`TEMP VIEW`) i analityka SQL.
3. **Ewolucja architektury:** Od Apache Hive do Delta Lake – dlaczego świat Big Data porzucił tradycyjne hurtownie Hive?
4. **Zapis do hurtowni Lakehouse:** Tabele zarządzane (`MANAGED`) i zewnętrzne (`EXTERNAL`) w Delta Lake vs pliki surowe (JSON).
5. **Poliglota formatów:** Zaawansowane złączenia heterogeniczne (łączenie tabeli Delta, widoków in-memory i plików JSON w jednym zapytaniu SQL).
6. **Fizyczna organizacja danych:** Partycjonowanie (`partitionBy`) na dysku oraz mechanizm **Partition Pruning** (obcinanie partycji).
7. **Nowoczesna alternatywa dla wiaderkowania:** Dlaczego tradycyjny `bucketBy` z Hive ustępuje miejsca clusteringowi i sortowaniu w Delta Lake.
8. **Możliwości Lakehouse:** Operacje modyfikacji na poziomie wiersza (`UPDATE` / `MERGE INTO`) oraz podróż w czasie (**Time Travel**).
9. **Rozbudowane zadania dla studentów:** Praktyczne ćwiczenia na danych sprzedażowych (`zamowienia.txt`).


## 1. Wymagania wstępne i konfiguracja środowiska

In [1]:
# Konfiguracja interpretera Pythona dla sterownika i wykonawców Sparka
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['PYSPARK_HOME'] = sys.executable


## 2. Inicjalizacja Sesji Spark z obsługą Delta Lake

W klasycznym podejściu z Apache Hive wymagane było wywołanie `.enableHiveSupport()` oraz konfiguracja silnika bazy Derby (`metastore_db`), co często powodowało problemy z blokowaniem plików na systemie Windows.

W nowoczesnej architekturze **Lakehouse** korzystamy z rozszerzenia **Delta Lake Catalog**:
* `spark.sql.extensions` $\rightarrow$ `io.delta.sql.DeltaSparkSessionExtension`: wstrzykuje do analizatora Sparka reguły obsługi tabel Delta,
* `spark.sql.catalog.spark_catalog` $\rightarrow$ `org.apache.spark.sql.delta.catalog.DeltaCatalog`: zastępuje domyślny katalog tabel katalogiem zgodnym z Delta Lake,
* `spark.sql.warehouse.dir` $\rightarrow$ katalog na dysku przeznaczony na składowanie tabel zarządzanych hurtowni danych.


In [2]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
import os

# Ścieżka do magazynu hurtowni danych (Lakehouse warehouse)
# możesz zmienić na tę z poprzedniego labu w celu dostępu do tych samych danych hurtiwni
warehouse_location = os.path.abspath("../data/lakehouse/warehouse")

# Inicjalizacja SparkSession z obsługą Delta Lake
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("PySpark_SQL_Delta_Lakehouse")
    .config("spark.jars.packages", "io.delta:delta-spark_2.13:4.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", warehouse_location)
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

# Ograniczenie nadmiarowych komunikatów logowania
spark.sparkContext.setLogLevel("ERROR")
print(f"Sesja Spark pomyślnie uruchomiona! Wersja Sparka: {spark.version}")


Sesja Spark pomyślnie uruchomiona! Wersja Sparka: 4.1.1


In [3]:
# Sprawdzenie aktualnego katalogu w Spark SQL
print("Aktywny katalog tabel:", spark.catalog.currentCatalog())


Aktywny katalog tabel: spark_catalog


## 3. Spark i SQL – Podstawy pracy z widokami tabelarycznymi

Apache Spark umożliwia zarejestrowanie dowolnego obiektu `DataFrame` jako widoku w pamięci podręcznej węzła sterującego (`createOrReplaceTempView`). 
Dzięki temu analitycy danych i inżynierowie mogą zadawać zapytania w standardowym dialekcie ANSI SQL bez konieczności przekształcania kodu w operacje funkcyjne PySpark.

Wczytajmy zbiór danych pracowników `employee.csv`.


In [ ]:
!pip install tqdm

In [6]:
from pathlib import Path
import csv
import random
import tqdm

employee_csv_path = "../data/employee.csv"
# liczba wierszy do stworzenia
n_rows = 10_000_000

# Zabezpieczenie: jeśli plik nie istnieje, wygenerujmy realistyczny zbiór testowy
if not os.path.exists(employee_csv_path):
    print("Plik employee.csv nie istnieje lokalnie. Generowanie danych testowych...")
    os.makedirs("../data", exist_ok=True)
    random.seed(42)
    firstnames = ["Zbigniew", "Krzysztof", "Wisława", "Mieczysław", "Agata", "Anna", "Piotr", "Tomasz", "Jan", "Katarzyna"]
    lastnames = ["Barański", "Kowalski", "Brzęczyszczykiewicz", "Pysla", "Nowak", "Wiśniewski", "Dąbrowski", "Lewandowski", "Wójcik", "Kamiński"]
    with open(employee_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "firstname", "lastname", "age", "salary"])
        for i in tqdm.tqdm(range(1, n_rows + 1), total=n_rows):
            writer.writerow([i, random.choice(firstnames), random.choice(lastnames), random.randint(22, 65), round(random.uniform(4500.0, 15000.0), 2)])
    print(f"Wygenerowano plik ../data/employee.csv z {n_rows} wierszy.")

df = spark.read.csv(employee_csv_path, header=True, inferSchema=True)
print(f"Liczba wczytanych rekordów pracowników: {df.count()}")
df.printSchema()
df.show(5)


Plik employee.csv nie istnieje lokalnie. Generowanie danych testowych...


100%|██████████| 10000000/10000000 [00:55<00:00, 179619.96it/s]


Wygenerowano plik ../data/employee.csv z 10000000 wierszy.
Liczba wczytanych rekordów pracowników: 10000000
root
 |-- id: integer (nullable = true)
 |-- firstname: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: double (nullable = true)

+---+---------+--------+---+--------+
| id|firstname|lastname|age|  salary|
+---+---------+--------+---+--------+
|  1|Krzysztof|Barański| 39| 7071.36|
|  2|  Wisława|Kowalski| 65|12277.01|
|  3|      Jan|Kowalski| 59| 8930.18|
|  4| Zbigniew|Kowalski| 35| 6942.94|
|  5|Katarzyna|Barański| 57|  6587.8|
+---+---------+--------+---+--------+
only showing top 5 rows


In [7]:
# Tworzymy widok tymczasowy w pamięci podręcznej Spark SQL
df.createOrReplaceTempView("EMPLOYEE_DATA")

# Wypisanie tabel zarejestrowanych w katalogu
# Zwróć uwagę na właściwość tableType ('TEMPORARY') oraz isTemporary (True)
spark.catalog.listTables()


[Table(name='employee_data', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [8]:
# Pobieranie danych za pomocą zapytania SQL
print("--- Pierwsze 4 rekordy ---")
spark.sql("SELECT * FROM EMPLOYEE_DATA LIMIT 4").show()

print("--- 10 imion pracowników ---")
spark.sql("SELECT firstname FROM EMPLOYEE_DATA LIMIT 10").show()


--- Pierwsze 4 rekordy ---
+---+---------+--------+---+--------+
| id|firstname|lastname|age|  salary|
+---+---------+--------+---+--------+
|  1|Krzysztof|Barański| 39| 7071.36|
|  2|  Wisława|Kowalski| 65|12277.01|
|  3|      Jan|Kowalski| 59| 8930.18|
|  4| Zbigniew|Kowalski| 35| 6942.94|
+---+---------+--------+---+--------+

--- 10 imion pracowników ---
+---------+
|firstname|
+---------+
|Krzysztof|
|  Wisława|
|      Jan|
| Zbigniew|
|Katarzyna|
|      Jan|
|    Agata|
|     Anna|
|     Anna|
|     Anna|
+---------+



In [9]:
# Agregacje wielowymiarowe: liczba pracowników i średnia pensja według imienia
spark.sql("""
    SELECT 
        firstname, 
        COUNT(firstname) AS pracownicy_count, 
        ROUND(AVG(salary), 2) AS avg_salary 
    FROM EMPLOYEE_DATA 
    GROUP BY firstname
    ORDER BY pracownicy_count DESC
""").show()


+----------+----------------+----------+
| firstname|pracownicy_count|avg_salary|
+----------+----------------+----------+
|  Zbigniew|         1001253|   9752.35|
|Mieczysław|         1001103|   9745.84|
|    Tomasz|         1000975|   9759.31|
|      Anna|         1000785|   9752.77|
|       Jan|         1000652|   9746.46|
|     Piotr|         1000101|   9747.28|
|   Wisława|          999478|   9747.98|
|     Agata|          998817|   9750.53|
| Katarzyna|          998806|   9753.81|
| Krzysztof|          998030|   9747.91|
+----------+----------------+----------+



In [10]:
# Zastosowanie parametrów w zapytaniu (symulacja 10% podwyżki płac)
the_raise = 0.10

query_raise = f"""
SELECT 
    firstname, 
    lastname, 
    salary, 
    ROUND(salary + salary * {the_raise}, 2) AS with_raise 
FROM EMPLOYEE_DATA 
LIMIT 5
"""

spark.sql(query_raise).show()


+---------+--------+--------+----------+
|firstname|lastname|  salary|with_raise|
+---------+--------+--------+----------+
|Krzysztof|Barański| 7071.36|    7778.5|
|  Wisława|Kowalski|12277.01|  13504.71|
|      Jan|Kowalski| 8930.18|    9823.2|
| Zbigniew|Kowalski| 6942.94|   7637.23|
|Katarzyna|Barański|  6587.8|   7246.58|
+---------+--------+--------+----------+



## 4. Ewolucja Hurtowni Danych: Od Apache Hive do Delta Lake

W klasycznym ekosystemie Big Data (Hadoop) centralnym elementem hurtowni danych było **Apache Hive** (stworzone przez Facebooka w 2007 r.).
Hive tłumaczyło zapytania Hive QL na serie zadań wsadowych MapReduce (lub Tez/Spark), a metadane tabel przechowywało w relacyjnej bazie (np. Derby lub MySQL).

### Ograniczenia Apache Hive:
1. **Brak prawdziwych transakcji ACID:** Równoczesny zapis i odczyt groził odczytem niespójnych danych (*dirty reads*).
2. **Brak efektywnych aktualizacji i kasowania wierszy:** Operacje `UPDATE` lub `DELETE` w Hive wymagały przepisywania całych partycji plików.
3. **Powolny Metastore:** Hive Metastore stawał się wąskim gardłem przy setkach tysięcy partycji.
4. **Brak wersjonowania i audytu:** Utrata lub przypadkowe nadpisanie danych wymagało odtwarzania z kopii zapasowej.

---

### Delta Lake jako fundament nowoczesnego Lakehouse:

| Cecha | Tradycyjny Apache Hive | Nowoczesne Delta Lake |
| :--- | :--- | :--- |
| **Format fizyczny** | Surowy ORC / Parquet / Text | Parquet + transakcyjny dziennik `_delta_log` |
| **Gwarancje ACID** | Brak lub ograniczone / niestabilne | Pełne transakcje ACID (poziom izolacji Serializable / WriteSerializable) |
| **Modyfikacje wierszowe** | Niedostępne lub bardzo kosztowne | Natywne operacje `UPDATE`, `DELETE`, `MERGE INTO` (Upsert) |
| **Wersjonowanie danych** | Brak | Pełne **Time Travel** (odczyt dowolnej wersji historycznej) |
| **Zarządzanie schematem** | Ręczne `ALTER TABLE` w metastore | *Schema Enforcement* i automatyczne *Schema Evolution* (`mergeSchema`) |
| **Optymalizacja odczytu** | Sztywne wiaderkowanie (`bucketBy`) | Dynamiczny Data Skipping (min/max statystyki) oraz Z-ORDER / Liquid Clustering |


In [11]:
from pyspark.sql.functions import monotonically_increasing_id

# Wzbogacamy ramkę o jednoznaczny unikalny identyfikator wiersza
df = df.withColumn("ID", monotonically_increasing_id())
df.show(5)


+---+---------+--------+---+--------+
| ID|firstname|lastname|age|  salary|
+---+---------+--------+---+--------+
|  0|Krzysztof|Barański| 39| 7071.36|
|  1|  Wisława|Kowalski| 65|12277.01|
|  2|      Jan|Kowalski| 59| 8930.18|
|  3| Zbigniew|Kowalski| 35| 6942.94|
|  4|Katarzyna|Barański| 57|  6587.8|
+---+---------+--------+---+--------+
only showing top 5 rows


In [12]:
# Podział zbioru danych na dwie części (30% i 70%)
splits = df.randomSplit(weights=[0.3, 0.7], seed=19)
print(f"Część 1: {splits[0].count()} wierszy, Część 2: {splits[1].count()} wierszy")

# Większa część trafia do nowego widoku tymczasowego w Spark SQL
splits[1].createOrReplaceTempView("EMPLOYEE_DATA_SPLIT_1")
spark.catalog.listTables()


Część 1: 2999012 wierszy, Część 2: 7000988 wierszy


[Table(name='employee_data', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='employee_data_split_1', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [13]:
import glob

# Mniejszą część zapisujemy do plików JSON na dysku
json_out_dir = "../data/json/employee"
splits[0].write.json(json_out_dir, mode='overwrite')

# Weryfikacja plików na dysku za pomocą Pythona (niezależnie od systemu operacyjnego)
json_files = glob.glob(f"{json_out_dir}/*.json")
print(f"Zapisano {len(json_files)} plików JSON w {json_out_dir}")
for p in json_files[:3]:
    print(" -", p)


Zapisano 3 plików JSON w ../data/json/employee
 - ../data/json/employee\part-00000-dc7bc679-891c-44f0-a48f-859b2002afbb-c000.json
 - ../data/json/employee\part-00001-dc7bc679-891c-44f0-a48f-859b2002afbb-c000.json
 - ../data/json/employee\part-00002-dc7bc679-891c-44f0-a48f-859b2002afbb-c000.json


In [14]:
from pyspark.sql.functions import col, lit, round

# Pobieramy losową próbkę 5% pracowników i dodajemy kolumny z premią/podwyżką
lucky_guys = (
    spark.sql("SELECT * FROM EMPLOYEE_DATA")
    .sample(fraction=0.05, seed=42)
    .withColumn('raise', lit('10%'))
    .withColumn('salary_after_raise', round(col('salary') * 1.10, 2))
)

print(f"Liczba wylosowanych pracowników z podwyżką: {lucky_guys.count()}")
lucky_guys.show(5)


Liczba wylosowanych pracowników z podwyżką: 499529
+---+---------+-------------------+---+--------+-----+------------------+
| id|firstname|           lastname|age|  salary|raise|salary_after_raise|
+---+---------+-------------------+---+--------+-----+------------------+
| 17|     Anna|Brzęczyszczykiewicz| 45| 8230.34|  10%|           9053.37|
| 80|   Tomasz|             Wójcik| 41|10922.62|  10%|          12014.88|
| 91|     Anna|           Kamiński| 57| 6054.98|  10%|           6660.48|
|106|    Agata|           Barański| 28| 10764.0|  10%|           11840.4|
|110| Zbigniew|             Wójcik| 56| 11712.0|  10%|           12883.2|
+---+---------+-------------------+---+--------+-----+------------------+
only showing top 5 rows


## 5. Zapis do Hurtowni Danych Lakehouse: Tabela Zarządzana w formacie Delta Lake

W starym podejściu tabelę zapisywano w Hive za pomocą:
`lucky_guys.write.saveAsTable("lucky_employees", format='parquet')`

W architekturze Lakehouse zapisujemy tabelę jako **tabelę Delta**:
`lucky_guys.write.format("delta").mode("overwrite").saveAsTable("lucky_employees")`

Spowoduje to:
1. Fizyczny zapis danych w kolumnowym formacie Parquet w podfolderze hurtowni `spark.sql.warehouse.dir/lucky_employees`.
2. Utworzenie transakcyjnego dziennika operacji `_delta_log/00000000000000000000.json`.
3. Zarejestrowanie tabeli o typie `MANAGED` w katalogu Spark Catalog.


In [15]:
# Zapisujemy próbkę jako trwałą tabelę Delta Lake w hurtowni
lucky_guys.write.format("delta").mode("overwrite").saveAsTable("lucky_employees")

# Zauważ: lucky_employees ma teraz tableType='MANAGED' (nie TEMPORARY!)
for tbl in spark.catalog.listTables():
    print(f"Tabela: {tbl.name:<25} Typ: {tbl.tableType:<12} Tymczasowa: {tbl.isTemporary}")


Tabela: lucky_employees           Typ: MANAGED      Tymczasowa: False
Tabela: employee_data             Typ: TEMPORARY    Tymczasowa: True
Tabela: employee_data_split_1     Typ: TEMPORARY    Tymczasowa: True


In [16]:

table_dir = os.path.join(warehouse_location, "lucky_employees")
print(f"Struktura katalogu tabeli Delta ({table_dir}):")

if os.path.exists(table_dir):
    for entry in sorted(os.listdir(table_dir)):
        entry_path = os.path.join(table_dir, entry)
        if os.path.isdir(entry_path):
            print(f" [KATALOG] {entry}/ (zawiera {len(os.listdir(entry_path))} plików)")
        else:
            print(f" [PLIK]    {entry} ({os.path.getsize(entry_path)} bajtów)")


Struktura katalogu tabeli Delta (C:\Users\Krzysztof\__projects\__2026_aiids_pyspark_3.11\data\lakehouse\warehouse\lucky_employees):
 [PLIK]    .part-00000-269e3028-1c62-4cba-ae1e-9db8aea873c4-c000.snappy.parquet.crc (23248 bajtów)
 [PLIK]    .part-00000-3214e294-b110-4137-97ca-ebc8b3a8c6ae-c000.snappy.parquet.crc (168 bajtów)
 [PLIK]    .part-00000-4ccf4a52-4eef-498b-94db-8cfb70bd8f22-c000.snappy.parquet.crc (3724 bajtów)
 [PLIK]    .part-00000-80e76511-edac-48a9-81b5-2e7c98e655a8-c000.snappy.parquet.crc (2944 bajtów)
 [PLIK]    .part-00001-2217162b-e45c-40bd-a2c8-58a0a4b88085-c000.snappy.parquet.crc (2944 bajtów)
 [PLIK]    .part-00001-730ae3cf-4fc0-4271-88d3-53ccf81e22d6-c000.snappy.parquet.crc (22972 bajtów)
 [PLIK]    .part-00002-82deed0d-a76c-4bf7-9f0a-ca3f1db010fa-c000.snappy.parquet.crc (20056 bajtów)
 [KATALOG] _delta_log/ (zawiera 17 plików)
 [PLIK]    part-00000-269e3028-1c62-4cba-ae1e-9db8aea873c4-c000.snappy.parquet (2974352 bajtów)
 [PLIK]    part-00000-3214e294-b110-4137-

## 6. Poliglota formatów: Złączenie heterogeniczne w Spark SQL

Jedną z największych zalet silnika Spark SQL w środowisku Lakehouse jest możliwość płynnego łączenia w jednym zapytaniu:
* **Tabel zarządzanych Delta Lake** (np. `lucky_employees`),
* **Widoków pamięciowych DataFrame** (np. `EMPLOYEE_DATA`),
* **Plików bezpośrednio z dysku** (np. `json.`../data/json/employee/``).


In [17]:
# Złączenie heterogeniczne: Delta Table + Memory View + JSON files
query_heterogeneous = """
SELECT 
    ed.ID, 
    ed.firstname, 
    ed.lastname, 
    ed.salary, 
    lucky.raise, 
    lucky.salary_after_raise
FROM json.`../data/json/employee/` AS jtable 
JOIN EMPLOYEE_DATA ed ON jtable.ID = ed.ID 
JOIN lucky_employees AS lucky ON ed.ID = lucky.ID
ORDER BY ed.ID
LIMIT 10
"""

print("Wynik złączenia heterogenicznego:")
spark.sql(query_heterogeneous).show()


Wynik złączenia heterogenicznego:
+---+----------+-------------------+--------+-----+------------------+
| ID| firstname|           lastname|  salary|raise|salary_after_raise|
+---+----------+-------------------+--------+-----+------------------+
|110|  Zbigniew|             Wójcik| 11712.0|  10%|           12883.2|
|186|       Jan|Brzęczyszczykiewicz| 6772.04|  10%|           7449.24|
|230|   Wisława|              Nowak|13748.89|  10%|          15123.78|
|348|   Wisława|Brzęczyszczykiewicz| 5022.51|  10%|           5524.76|
|506|Mieczysław|Brzęczyszczykiewicz|14957.68|  10%|          16453.45|
|586|Mieczysław|           Kamiński| 5454.89|  10%|           6000.38|
|684|     Piotr|           Kowalski|14004.68|  10%|          15405.15|
|688|     Piotr|              Nowak|  7647.4|  10%|           8412.14|
|710| Katarzyna|          Dąbrowski|14640.03|  10%|          16104.03|
|738|       Jan|Brzęczyszczykiewicz| 6252.09|  10%|            6877.3|
+---+----------+-------------------+-------

## 7. Fizyczna Organizacja Danych: Partycjonowanie (`partitionBy`) i Partition Pruning

### Kiedy stosować partycjonowanie?
Partycjonowanie dzieli tabelę na dysku na fizyczne podkatalogi oparte na wartościach wybranej kolumny (np. `lastname=Kowalski/part-0.parquet`).
* **Zalecane:** Kolumny o **niskiej lub średniej kardynalności** (np. rok, miesiąc, kraj, status), które często występują w klauzulach `WHERE`.
* **Niezalecane:** Kolumny o bardzo wysokiej kardynalności (np. unikalne ID, PESEL, data z godziną) – prowadzi to do problemu *Small Files Problem* (tysiące mikroskopijnych plików przeciążających system plików i pamięć sterownika).

### Co to jest Partition Pruning (obcinanie partycji)?
Gdy zadajemy zapytanie z warunkiem `WHERE lastname = 'Pysla'`, silnik Sparka **nie skanuje całego dysku**, lecz wczytuje wyłącznie pliki znajdujące się w folderze `lastname=Pysla/`. Pozostałe partycje są całkowicie pomijane (*pruned*).


In [18]:
# Zapis tabeli Delta Lake z fizycznym podziałem na partycje po kolumnie lastname
df.write.format("delta").partitionBy("lastname").mode("overwrite").saveAsTable("employees_partitioned_lastname")

print("Tabela employees_partitioned_lastname została pomyślnie zapisana!")


Tabela employees_partitioned_lastname została pomyślnie zapisana!


In [19]:
part_dir = os.path.join(warehouse_location, "employees_partitioned_lastname")
print(f"Podkatalogi partycji na dysku w {part_dir}:")

subdirs = [d for d in os.listdir(part_dir) if os.path.isdir(os.path.join(part_dir, d)) and d != "_delta_log"]
for sd in sorted(subdirs)[:8]:
    file_count = len(os.listdir(os.path.join(part_dir, sd)))
    print(f" - {sd:<30} (liczba plików parquet: {file_count})")


Podkatalogi partycji na dysku w C:\Users\Krzysztof\__projects\__2026_aiids_pyspark_3.11\data\lakehouse\warehouse\employees_partitioned_lastname:
 - lastname=Barański              (liczba plików parquet: 12)
 - lastname=Brzęczyszczykiewicz   (liczba plików parquet: 12)
 - lastname=Dąbrowski             (liczba plików parquet: 12)
 - lastname=Kamiński              (liczba plików parquet: 12)
 - lastname=Kowalski              (liczba plików parquet: 12)
 - lastname=Lewandowski           (liczba plików parquet: 12)
 - lastname=Nowak                 (liczba plików parquet: 12)
 - lastname=Pysla                 (liczba plików parquet: 12)


In [20]:
# Porównanie planów wykonania zapytań (EXPLAIN)

print("=== 1. PLAN DLA TABELI ZWYKŁEJ (Brak partycjonowania - pełny skan) ===")
spark.sql("SELECT lastname, AVG(salary) FROM lucky_employees WHERE lastname = 'Pysla' GROUP BY lastname").explain(mode='formatted')

print("\n=== 2. PLAN DLA TABELI PARTITIONED (Partition Pruning aktywny) ===")
# Zwróć uwagę na sekcję PartitionFilters w planie wykonania!
spark.sql("SELECT lastname, AVG(salary) FROM employees_partitioned_lastname WHERE lastname = 'Pysla' GROUP BY lastname").explain(mode='formatted')


=== 1. PLAN DLA TABELI ZWYKŁEJ (Brak partycjonowania - pełny skan) ===
== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Filter (2)
            +- Scan parquet spark_catalog.default.lucky_employees (1)


(1) Scan parquet spark_catalog.default.lucky_employees
Output [2]: [lastname#3075, salary#3077]
Batched: true
Location: PreparedDeltaFileIndex [file:/C:/Users/Krzysztof/__projects/__2026_aiids_pyspark_3.11/data/lakehouse/warehouse/lucky_employees]
PushedFilters: [IsNotNull(lastname), EqualTo(lastname,Pysla)]
ReadSchema: struct<lastname:string,salary:double>

(2) Filter
Input [2]: [lastname#3075, salary#3077]
Condition : (isnotnull(lastname#3075) AND (lastname#3075 = Pysla))

(3) HashAggregate
Input [2]: [lastname#3075, salary#3077]
Keys [1]: [lastname#3075]
Functions [1]: [partial_avg(salary#3077)]
Aggregate Attributes [2]: [sum#3149, count#3150L]
Results [3]: [lastname#3075, sum#3151, count#3152L]

(4) Exchange
Inp

In [32]:
%%time
# Pomiar czasu wykonania zapytania na tabeli pełnej
spark.sql("""
    SELECT lastname, ROUND(AVG(salary), 2) AS avg_salary, COUNT(*) AS count
    FROM EMPLOYEE_DATA 
    WHERE lastname = 'Pysla' 
    GROUP BY lastname
""").show()

+--------+----------+------+
|lastname|avg_salary| count|
+--------+----------+------+
|   Pysla|   9746.23|100002|
+--------+----------+------+

CPU times: total: 15.6 ms
Wall time: 1.48 s


In [33]:
%%time
# Pomiar czasu wykonania zapytania na tabeli partycjonowanej
spark.sql("""
    SELECT lastname, ROUND(AVG(salary), 2) AS avg_salary, COUNT(*) AS count
    FROM employees_partitioned_lastname 
    WHERE lastname = 'Pysla' 
    GROUP BY lastname
""").show()


+--------+----------+------+
|lastname|avg_salary| count|
+--------+----------+------+
|   Pysla|   9746.23|100002|
+--------+----------+------+

CPU times: total: 0 ns
Wall time: 932 ms


## 8. Od Wiaderkowania (`bucketBy` w Hive) do Nowoczesnego Liquid Clustering (`CLUSTER BY` w Delta Lake)

W klasycznym Hive stosowano podział na tzw. **wiaderka (buckets)**:
```python
# Stare, przestarzałe podejście z Hive:
df.write.bucketBy(16, 'ID').sortBy('salary').saveAsTable('employee_id_bucketed')
```

### Dlaczego wiaderkowanie w Hive odeszło do lamusa?
1. **Sztywność struktury:** Liczba wiaderek (np. 16) była ustalana na stałe podczas tworzenia tabeli i nie mogła być dynamicznie zmieniana wraz ze wzrostem wielkości danych.
2. **Brak wsparcia dla ewolucji schematu:** Wszelkie zmiany strukturalne lub zmiana klucza wiaderkowania wymagały ręcznego przepisywania całej tabeli od zera!
3. **Słaba integracja poza Hive:** Wiaderkowanie działało wyłącznie w metastore Hive; odczyt plików bezpośrednio przez Sparka z reguły nie potrafił wykorzystać podziału na buckety bez pełnych metadanych Hive.

---

### Liquid Clustering w Delta Lake (Delta 3.x+ / `CLUSTER BY`)
W technologii Delta Lake wprowadzono mechanizm **Liquid Clustering**, który zastępuje zarówno tradycyjne partycjonowanie (`partitionBy`), jak i wiaderkowanie (`bucketBy`):
* **Brak problemu małych plików:** Dane składowane są w elastyczny sposób w jednym katalogu (zamiast tworzenia setek podkatalogów partycji).
* **Wielowymiarowy Data Skipping:** Rekordy są grupowane w plikach z zachowaniem lokalności przestrzennej wzdłuż kilku wybranych kolumn (np. nazwisko i wiek).
* **Elastyczność w locie (`ALTER TABLE`):** Możesz w dowolnym momencie zmienić kolumny klastrowania **bez przepisywania istniejących danych**!
* **Inkrementalna optymalizacja:** Polecenie `OPTIMIZE` automatycznie porządkuje tylko nowo dodane dane.


In [21]:
# 1. Tworzenie tabeli Delta z Liquid Clustering (CLUSTER BY)
# Klastrujemy dane jednocześnie po kolumnie 'lastname' oraz 'age'
spark.sql("""
CREATE OR REPLACE TABLE employees_clustered
USING DELTA
CLUSTER BY (lastname, age)
AS SELECT * FROM EMPLOYEE_DATA
""")

print("Tabela employees_clustered została pomyślnie utworzona z Liquid Clustering!")


Tabela employees_clustered została pomyślnie utworzona z Liquid Clustering!


In [22]:
# 2. Weryfikacja metadanych klastrowania w katalogu Delta Lake
# Zwróć uwagę na kolumnę 'clusteringColumns' w wynikach DESCRIBE DETAIL
spark.sql("""
    DESCRIBE DETAIL employees_clustered
""").select("format", "clusteringColumns", "numFiles", "sizeInBytes").show(truncate=False)


+------+-----------------+--------+-----------+
|format|clusteringColumns|numFiles|sizeInBytes|
+------+-----------------+--------+-----------+
|delta |[lastname, age]  |3       |113567640  |
+------+-----------------+--------+-----------+



In [23]:
# 3. Uruchomienie procedury optymalizacji Liquid Clustering (OPTIMIZE)
# OPTIMIZE przeorganizowuje pliki Parquet tak, aby zminimalizować zakresy min-max w klastrach
print("Uruchamianie optymalizacji clusteringu...")
optimize_result = spark.sql("OPTIMIZE employees_clustered")
optimize_result.select("path", "metrics.numFilesAdded", "metrics.numFilesRemoved").show(truncate=False)


Uruchamianie optymalizacji clusteringu...
+----------------------------------------------------------------------------------------------------------+-------------+---------------+
|path                                                                                                      |numFilesAdded|numFilesRemoved|
+----------------------------------------------------------------------------------------------------------+-------------+---------------+
|file:/C:/Users/Krzysztof/__projects/__2026_aiids_pyspark_3.11/data/lakehouse/warehouse/employees_clustered|1            |3              |
+----------------------------------------------------------------------------------------------------------+-------------+---------------+



### Dynamiczna zmiana kolumn klastrowania (`ALTER TABLE ... CLUSTER BY`)

W klasycznym Hive lub przy tradycyjnym `partitionBy` zmiana kolumny partycjonującej/wiaderkującej wymagała pełnego przepisania całej tabeli (często terabajtów danych). 

W Delta Lake z Liquid Clustering zmiana kluczy klastrowania jest operacją metadanową w `_delta_log` i trwa ułamek sekundy:


In [24]:
# 4. Zmiana kolumn klastrowania w locie (np. z (lastname, age) na (lastname, salary))
spark.sql("""
    ALTER TABLE employees_clustered 
    CLUSTER BY (lastname, salary)
""")

# Weryfikacja: zaktualizowana definicja kolumn klastrowania
print("Zaktualizowana definicja clusteringColumns:")
spark.sql("DESCRIBE DETAIL employees_clustered").select("format", "clusteringColumns").show(truncate=False)

# Każde kolejne wywołanie OPTIMIZE będzie układać nowe pliki według nowej definicji!


Zaktualizowana definicja clusteringColumns:
+------+------------------+
|format|clusteringColumns |
+------+------------------+
|delta |[lastname, salary]|
+------+------------------+



In [25]:
# 5. Odpytanie tabeli sklastrowanej z filtrem
# Silnik Delty wykorzystuje statystyki min-max wygenerowane przez Liquid Clustering
spark.sql("""
    SELECT firstname, lastname, age, salary 
    FROM employees_clustered 
    WHERE lastname = 'Pysla' AND salary > 8000
    ORDER BY salary DESC
    LIMIT 5
""").show()


+----------+--------+---+--------+
| firstname|lastname|age|  salary|
+----------+--------+---+--------+
|       Jan|   Pysla| 27|14999.99|
|Mieczysław|   Pysla| 49|14999.99|
| Katarzyna|   Pysla| 22|14999.98|
|  Zbigniew|   Pysla| 60|14999.96|
| Katarzyna|   Pysla| 27|14999.96|
+----------+--------+---+--------+



## 9. Cechy Lakehouse, których brakowało w Hive: Transakcje ACID, Modyfikacje Wierszowe i Time Travel

W klasycznym Data Lake modyfikacja pojedynczego wiersza wymagała przepisania całej tabeli lub całej partycji.
W Delta Lake mamy do dyspozycji:
1. **Operacje `UPDATE` / `DELETE` / `MERGE INTO` (Upsert)** – atomowe transakcje wierszowe.
2. **Time Travel (Podróż w czasie)** – możliwość odpytania stanu tabeli z dowolnej historycznej wersji lub znacznika czasu.


In [ ]:
# Pobierzmy ID pierwszego pracownika z tabeli lucky_employees
sample_id = spark.table("lucky_employees").select("ID", "salary").first()["ID"]
old_salary = spark.table("lucky_employees").filter(f"ID = {sample_id}").first()["salary"]
print(f"Pracownik ID={sample_id}: aktualna pensja = {old_salary}")

# Wykonajmy polecenie SQL UPDATE (niemożliwe w tradycyjnym Hive bez transakcyjnego silnika ACID)
spark.sql(f"""
    UPDATE lucky_employees 
    SET salary = 99999.00, salary_after_raise = 99999.00 
    WHERE ID = {sample_id}
""")

# Weryfikacja po aktualizacji
new_salary = spark.table("lucky_employees").filter(f"ID = {sample_id}").first()["salary"]
print(f"Pracownik ID={sample_id}: nowa pensja po transakcji UPDATE = {new_salary}")


In [ ]:
# Wyświetlenie historii transakcji tabeli w dzienniku _delta_log
print("--- Historia transakcji tabeli lucky_employees ---")
spark.sql("DESCRIBE HISTORY lucky_employees").select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

# Time Travel: odczyt wersji pierwotnej (version = 0) sprzed operacji UPDATE
print("--- Stan rekordu w wersji 0 (sprzed aktualizacji) ---")
spark.sql(f"""
    SELECT ID, firstname, lastname, salary 
    FROM lucky_employees VERSION AS OF 0 
    WHERE ID = {sample_id}
""").show()

print("--- Stan rekordu w wersji bieżącej (po aktualizacji) ---")
spark.sql(f"""
    SELECT ID, firstname, lastname, salary 
    FROM lucky_employees 
    WHERE ID = {sample_id}
""").show()


## 10. Zadania dla studentów

Poniższe zadania sprawdzają umiejętność:
* Czyszczenia i normalizacji danych tekstowych w PySpark DataFrame,
* Konwersji typów danych i radzenia sobie z uszkodzonym kodowaniem znaków,
* Projektowania tabel w architekturze **Lakehouse** z partycjonowaniem w **Delta Lake**,
* Analizy planów zapytań (**Partition Pruning**),
* Złożonych heterogenicznych zapytań SQL łączących wiele formatów składowania.


### Zadanie 1: Czyszczenie, normalizacja typów i klucze w zbiorze `zamowienia.txt`

W repozytorium znajduje się plik `zamowienia.txt` (lub `../data/zamowienia.txt`).
Wczytaj ten plik za pomocą Sparka i dokonaj niezbędnych transformacji czyszczących:

1. **Format pliku i schemat:** Plik jest rozdzielany średnikami (`;`) i posiada nagłówek:
   `Kraj;Sprzedawca;Data zamowienia;idZamowienia;Utarg`
2. **Naprawa uszkodzonego kodowania znaków:** W kolumnie `Sprzedawca` występują uszkodzone litery (np. `Sowiäski` $\rightarrow$ `Sowiński`). Zamień zniekształcone znaki na poprawne polskie litery.
3. **Oczyszczenie i konwersja kolumny `Utarg`:** Wartości zapisane są jako tekst, np. `"1 863,40 zˆ"` lub `"440,00 zˆ"`. Należy:
   * Usunąć spacje (separatory tysięcy),
   * Zamienić przecinek na kropkę,
   * Usunąć jednostkę walutową (`zˆ` / `zł`),
   * Rzutować kolumnę na poprawny typ numeryczny (`DoubleType` lub `DecimalType(10, 2)`).
4. **Konwersja daty:** Kolumnę `Data zamowienia` (np. `16.07.2003`) skonwertuj na właściwy typ `DateType` za pomocą formatu `'dd.MM.yyyy'`.
5. **Klucz główny / indeks:** Upewnij się, że `idZamowienia` jest poprawnym typem całkowitym (`IntegerType`), oraz opcjonalnie wygeneruj unikalny klucz surogatowy za pomocą `monotonically_increasing_id()`.

---

#### Wskazówki i przydatna dokumentacja:
* **Wczytanie pliku CSV ze średnikiem:** [`spark.read.option("sep", ";").option("header", "true").csv(...)`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.csv.html).
* **Czyszczenie tekstu i regex:** Wykorzystaj [`regexp_replace()`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.regexp_replace.html) z `pyspark.sql.functions` do usunięcia spacji i znaków waluty oraz [`replace()`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Column.replace.html) / [`when()`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.when.html) do korekty nazwisk.
* **Konwersja daty:** Zastosuj funkcję [`to_date(col("Data zamowienia"), "dd.MM.yyyy")`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.to_date.html).
* **Rzutowanie typów:** Wykorzystaj metodę [`.cast("double")`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Column.cast.html).


In [ ]:
# TODO: Miejsce na Twoje rozwiązanie Zadania 1

from pyspark.sql.functions import col, regexp_replace, to_date, when, monotonically_increasing_id

# Ścieżka do pliku (obsługa lokalizacji w notebooks/ lub data/)
orders_path = "zamowienia.txt" if os.path.exists("zamowienia.txt") else "../data/zamowienia.txt"

# 1. Wczytanie pliku


# 2. Czyszczenie Sprzedawca, Utarg oraz Data zamowienia


# 3. Wyświetlenie schematu i 10 pierwszych rekordów


### Zadanie 2: Partycjonowanie w hurtowni Delta Lake i analiza Partition Pruning

Wykorzystując oczyszczoną ramkę danych z Zadania 1:

1. **Zapis do tabeli Delta z partycjonowaniem:** Zapisz ramkę jako trwałą tabelę w hurtowni danych Delta Lake pod nazwą `zamowienia_delta_kraj`, partycjonując dane po kolumnie `Kraj` (`partitionBy("Kraj")`).
2. **Weryfikacja fizyczna na dysku:** Sprawdź w języku Python (`os.listdir`), jakie foldery partycji powstały w katalogu hurtowni (`warehouse_location/zamowienia_delta_kraj`).
3. **Analiza planu wykonania (Partition Pruning):**
   * Wykonaj zapytanie agregujące (np. sumaryczny utarg i liczbę zamówień) z filtrem `WHERE Kraj = 'Polska'` na tabeli **niepartycjonowanej** (zwykły widok lub tabela bez podziału).
   * Wykonaj to samo zapytanie na tabeli **partycjonowanej** `zamowienia_delta_kraj`.
   * Porównaj plany zapytań za pomocą `.explain()` – zidentyfikuj sekcję `PartitionFilters` wskazującą na obcięcie partycji.
4. **Wnioski:** Skomentuj krótko w markdown: dlaczego przy małym zbiorze danych (800 wierszy) czas wykonania nie musi być niższy, a dlaczego zysk pojawia się dopiero przy wolumenach liczonych w gigabajtach/terabajtach.

---

#### Wskazówki i przydatna dokumentacja:
* **Zapis tabeli partycjonowanej w Delcie:** [`df.write.format("delta").partitionBy("Kraj").mode("overwrite").saveAsTable("zamowienia_delta_kraj")`](https://docs.delta.io/latest/delta-batch.html#partition-data).
* **Plan wykonania:** Wywołaj [`.explain()`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.explain.html) na obiekcie wynikowym zapytania `spark.sql(...)`.


In [ ]:
# TODO: Miejsce na Twoje rozwiązanie Zadania 2

# 1. Zapis z partycjonowaniem do tabeli Delta


# 2. Sprawdzenie katalogów partycji na dysku


# 3. Porównanie explain() dla tabeli zwykłej i partycjonowanej


### Zadanie 3: Heterogeniczne złączenie 4 źródeł w Lakehouse i audyt w Delta Lake

Wykorzystując oczyszczone dane zamówień, przygotuj 4 różne podzbiory próbek (lub podziały losowe) i wzbogać każdy z nich o jedną nową cechę:

1. **Zbiór 1 (Miesiąc):** Dodaj kolumnę `miesiac` (wyciągnięty numer miesiąca z daty zamówienia za pomocą funkcji `month()`). Zarejestruj jako widok tymczasowy w pamięci: `temp_zamowienia_miesiac`.
2. **Zbiór 2 (Wartość netto):** Dodaj kolumnę `utarg_netto` (obliczoną jako `round(Utarg / 1.23, 2)` przy założeniu 23% VAT). Zapisz ten zbiór jako tabelę **Delta Lake** w hurtowni (`zamowienia_delta_netto`).
3. **Zbiór 3 (Nazwisko wielkimi literami):** Dodaj kolumnę `sprzedawca_upper` (funkcja `upper(col("Sprzedawca"))`). Zapisz zbiór jako pliki w formacie **Parquet** (`../data/parquet/zamowienia_upper`).
4. **Zbiór 4 (Waluta):** Dodaj kolumnę `waluta` ze stałą wartością `'PLN'` (funkcja `lit('PLN')`). Zapisz zbiór jako pliki w formacie **JSON** (`../data/json/zamowienia_waluta`).

**Zapytanie SQL (Poliglota Lakehouse):**
Napisz jedno zapytanie w Spark SQL, które złączy wszystkie 4 źródła po kluczu `idZamowienia` i wyświetli:
* `idZamowienia`, `Kraj`, `Sprzedawca`, `Data_zamowienia`, `Utarg`,
* Oraz wszystkie 4 nowo utworzone kolumny: `miesiac`, `utarg_netto`, `sprzedawca_upper`, `waluta`.

> [!TIP]
> W zapytaniu SQL możesz odwoływać się bezpośrednio do plików na dysku za pomocą składni:
> `parquet.`ścieżka`` oraz `json.`ścieżka``, a do tabeli Delty poprzez nazwę zarejestrowanej tabeli lub `delta.`ścieżka``.

---

#### Wskazówki i przydatna dokumentacja:
* **Funkcje daty i tekstu:** [`month()`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.month.html), [`upper()`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.upper.html), [`lit()`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.lit.html).
* **Złączenia w SQL:** Użyj [`JOIN`](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-join.html) (lub `LEFT JOIN` jeśli próbki nie zawierają identycznego zbioru kluczy).


In [ ]:
# TODO: Miejsce na Twoje rozwiązanie Zadania 3

from pyspark.sql.functions import month, upper, lit, round

# 1. Przygotowanie i zapis 4 zbiorów (widok in-memory, Delta, Parquet, JSON)


# 2. Heterogeniczne zapytanie SQL łączące 4 formaty


## 11. Zatrzymanie sesji Spark

Po zakończeniu pracy zwalniamy zasoby klastra lokalnego.


In [ ]:
# Zatrzymanie sesji SparkSession
spark.stop()
print("Sesja SparkContext została pomyślnie zamknięta.")
